In [1]:
# mask-analog imports
import run_experiments as rexp
# create an Experiments object which allows you to access previous experiments and define new ones (and train a mask)
exp_object = rexp.Experiments('experiments.json')
import plot_analogs as pa
import best_analogs as ba
import numpy as np
# hide some warnings that are not helpful
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
# make plots show up in the notebook
%matplotlib inline

# workflow for a new experiment

#### 1. preprocess data (saves data that is temporally and spatially smoothed)

#### 2. create the settings for a new experiment

#### 3. train (and plot) a mask based on the settings

#### 4. generate (or select a previously generated) analog library

#### 5. find the best analogs for a given state of interest (model, member)

### 1. preprocess data (saves data that is temporally and spatially smoothed)

In [ ]:
# var_list and model_list = -1 means use all available
# scratch means load the models and resolutions by reading the files directly (otherwise, use the version I hand put in there)

# uncomment and run this if you need something that is not already preprocessed -- will take a while (several hours)
# import data_processing as dp
# dp.preprocess_data(resolution, var_list=-1, model_list=-1, overwrite_data=False, scratch=False)

### 2. create the settings for a new experiment

In [ ]:
# print out the settings
# print(exp_object.make_exp_dictionary.__doc__)
# print('---------------------------------------------------------')
# this lists all the regions available (you can add new regions to regions.json)
# exp_object.list_regions

In [ ]:
experiment_name = "exp_name"
region = "region_name"
variable = ['tas']
leadtime = 5
all_models = ['GISS-E2-2-G', 'NorESM2-LM', 'ACCESS-ESM1-5', 'CanESM5-1', 'CMCC-CM2-HR4', 'CESM2', 'AWI-ESM-1-1', 'CAMS-CSM1', 'CMCC-ESM2', 'CESM2-WACCM', 'MPI-ESM1-2-HR', 'AWI-CM-1-1', 'MIROC', 'MPI-ESM1-2-LR', 'GISS-E2-1-G', 'CNRM-CM6-1', 'CMCC-CM2-SR5', 'MRI-ESM2-0', 'MIROC-ES2H', 'CanESM5-CanOE', 'ACCESS-CM2', 'BCC-ESM1', 'UKESM1-1-LL', 'UKESM1-0-LL', 'CanESM5', 'BCC-CSM2', 'TaiESM', 'NorESM2-MM', 'MIROC-ES2L']

In [ ]:
# now, define a new experiment with all the settings you want to change
new_exp = exp_object.make_exp_dictionary(experiment_name, variable_list=variable, analog_models=all_models, soi_models=all_models, leadtime=leadtime, split=[5, 3, 2], region=region, shapefile=None, val_models=all_models, target_var=variable, window=False, trajectory=1, resolutions=1, time_range=False, norm_data=True, baseline=[1961, 1990], remove_trend=False, norm_weights=True, transfer_exp=None, mask=False, prediction_nodes=[], prediction_activation='elu', learning_rate=0.001, loss='mse', patience=50, min_delta=0.005, batch_size=64, val_batch_size=2500, max_epochs=5000, mask_activation='relu', mask_init="ones", mask_l1=0.0, mask_l2=0.0, seed=20)

# this adds the experiment to the json file
exp_object.add_experiment(new_exp)

### 3. train (and plot) a mask based on the settings

In [ ]:
# NOTE: this may need to be run in a terminal when the amount of data is large
exp_object.make_mask(experiment_name, overwrite_model=True, overwrite_mask=True)

In [ ]:
# plot the mask
exp_object.plot_mask(experiment_name, colorbar=False)
# # optional -- make a gif of the training
# exp_object.make_mask_gif(experiment_name)

### 4. generate (or select a previously generated) analog library

In [ ]:
# model_list is to set the models in the library, n_members sets how many (-1 uses all)
# library_name can be the name of a previous experiment and will use that library
library = ba.BestAnalogs(experiment_name, library_name=None, model_list=all_models+['BEST'], n_members=-1)

### 5. find the best analogs for a given state of interest (model, member)

In [ ]:
# this ranks the analogs in the library for this SOI
# this may also need to be run in the terminal
tether = 2
soi_model = 'BEST'
soi_member = 0
library.get_all_soi_analogs(soi_model, soi_member, tether=tether, savename=None, quant=None, mask_type='weights', leadtime=None)

# for global or regional masks
# library.get_all_soi_analogs(soi_model, soi_member, tether=tether, savename='global', quant=None, mask_type='global', leadtime=None)
# library.get_all_soi_analogs(soi_model, soi_member, tether=tether, savename='regional', quant=None, mask_type='regional', leadtime=None)